# Chapter VI — Big O

Big O describes how the running time (or memory) of an algorithm **grows** as the input grows. It is the language used to compare every solution in the rest of this repo.

Key rules:
1. **Drop constants** — `O(2N)` is `O(N)`.
2. **Drop non-dominant terms** — `O(N² + N)` is `O(N²)`.
3. **Add vs. multiply** — do A then B → `O(A + B)`; for each A do B → `O(A · B)`.
4. **Amortized time** — a dynamic array's append is `O(1)` amortized even though a resize costs `O(N)`.
5. **log N runtimes** — when the problem space halves each step (binary search, balanced trees).
6. **Recursive runtimes** — often `O(branches^depth)`.

This notebook measures a few of these empirically.

In [1]:
import time
from collections.abc import Callable


def time_it(fn: Callable[[int], object], sizes: list[int]) -> None:
    """Print how long fn(n) takes for each n, plus the ratio to the previous size."""
    prev = None
    for n in sizes:
        start = time.perf_counter()
        fn(n)
        elapsed = time.perf_counter() - start
        ratio = f"x{elapsed / prev:5.1f}" if prev else ""
        print(f"n={n:>8}: {elapsed * 1000:8.3f} ms {ratio}")
        prev = elapsed

## O(N) vs O(N²)
Doubling N should roughly double linear work and quadruple quadratic work.

In [2]:
def linear(n: int) -> int:
    return sum(range(n))


def quadratic(n: int) -> int:
    total = 0
    for i in range(n):
        for j in range(n):
            total += 1
    return total


print("linear");    time_it(linear, [250_000, 500_000, 1_000_000, 2_000_000])
print("quadratic"); time_it(quadratic, [250, 500, 1000, 2000])

linear
n=  250000:    3.757 ms 
n=  500000:    6.447 ms x  1.7
n= 1000000:   10.258 ms x  1.6
n= 2000000:   16.271 ms x  1.6
quadratic
n=     250:    0.753 ms 
n=     500:    3.433 ms x  4.6
n=    1000:   13.615 ms x  4.0
n=    2000:   47.155 ms x  3.5


## Amortized time: dynamic arrays
Python's `list` over-allocates when it grows. We can watch the capacity jumps via `sys.getsizeof`. Most appends cost nothing extra; occasionally one triggers a copy. The total cost of N appends is `O(N)`, i.e. `O(1)` amortized.

In [3]:
import sys

items: list[int] = []
last_size = sys.getsizeof(items)
resizes = []
for i in range(200):
    items.append(i)
    size = sys.getsizeof(items)
    if size != last_size:
        resizes.append(len(items))
        last_size = size
print("list resized at lengths:", resizes)

list resized at lengths: [1, 5, 9, 17, 25, 33, 41, 53, 65, 77, 93, 109, 129, 149, 173]


## Our own dynamic array (to make the amortization explicit)

In [4]:
class DynamicArray:
    def __init__(self) -> None:
        self._capacity = 1
        self._size = 0
        self._data: list[object] = [None]
        self.copies = 0  # count element copies to show amortized cost

    def append(self, value: object) -> None:
        if self._size == self._capacity:
            self._resize(self._capacity * 2)
        self._data[self._size] = value
        self._size += 1

    def _resize(self, new_capacity: int) -> None:
        new_data: list[object] = [None] * new_capacity
        for i in range(self._size):
            new_data[i] = self._data[i]
            self.copies += 1
        self._data, self._capacity = new_data, new_capacity

    def __getitem__(self, i: int) -> object:
        if not 0 <= i < self._size:
            raise IndexError(i)
        return self._data[i]

    def __len__(self) -> int:
        return self._size


arr = DynamicArray()
n = 1_000_000
for i in range(n):
    arr.append(i)
print(f"{n} appends caused {arr.copies} copies -> {arr.copies / n:.2f} copies per append")
assert arr.copies < 2 * n

1000000 appends caused 1048575 copies -> 1.05 copies per append


## log N: binary search

In [5]:
def binary_search(xs: list[int], target: int) -> int:
    lo, hi, steps = 0, len(xs) - 1, 0
    while lo <= hi:
        steps += 1
        mid = (lo + hi) // 2
        if xs[mid] == target:
            return steps
        if xs[mid] < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return steps


for n in [1_000, 1_000_000, 1_000_000_000]:
    print(f"n={n:>13,}: at most {binary_search(range(n), -1)} steps")  # range supports indexing

n=        1,000: at most 9 steps
n=    1,000,000: at most 19 steps
n=1,000,000,000: at most 29 steps


## Recursive runtimes: O(2^N) vs memoized O(N)
The naive Fibonacci tree has two branches and depth N → `O(2^N)`. Caching collapses it to `O(N)`.

In [6]:
from functools import cache

calls = 0


def fib_naive(n: int) -> int:
    global calls
    calls += 1
    return n if n <= 1 else fib_naive(n - 1) + fib_naive(n - 2)


@cache
def fib_memo(n: int) -> int:
    return n if n <= 1 else fib_memo(n - 1) + fib_memo(n - 2)


for n in [10, 20, 25]:
    calls = 0
    fib_naive(n)
    print(f"fib({n}) naive calls: {calls:,}")
print("fib_memo(300) =", fib_memo(300))

fib(10) naive calls: 177
fib(20) naive calls: 21,891
fib(25) naive calls: 242,785
fib_memo(300) = 222232244629420445529739893461909967206666939096499764990979600


## Examples & exercises (paraphrased)
Classic "what's the runtime?" snippets, with answers as comments.

In [7]:
def ex_print_pairs(arr):          # O(N^2): nested loops over same array
    for a in arr:
        for b in arr:
            pass


def ex_two_arrays(a, b):          # O(A*B): different inputs -> different variables
    for x in a:
        for y in b:
            pass


def ex_reverse(arr):              # O(N): the loop runs N/2 times, constants dropped
    for i in range(len(arr) // 2):
        arr[i], arr[-1 - i] = arr[-1 - i], arr[i]


def ex_is_prime(n):               # O(sqrt N)
    i = 2
    while i * i <= n:
        if n % i == 0:
            return False
        i += 1
    return n >= 2


def ex_permutations(s, prefix=""):  # O(N^2 * N!): N! leaves, each path O(N), string ops O(N)
    if not s:
        return [prefix]
    return [p for i in range(len(s)) for p in ex_permutations(s[:i] + s[i + 1:], prefix + s[i])]


def ex_sum_digits(n):              # O(log N): number of digits
    total = 0
    while n > 0:
        total += n % 10
        n //= 10
    return total


def ex_power(a, b):                # O(b) naive; see Chapter 8 for O(log b)
    return 1 if b == 0 else a * ex_power(a, b - 1)


assert ex_is_prime(97) and not ex_is_prime(91)
assert len(ex_permutations("abcd")) == 24
assert ex_sum_digits(1234) == 10
assert ex_power(2, 10) == 1024
print("all good")

all good
